# Optimal control 

In [13]:
#import tensorflow.compat.v1 as tf
#tf.disable_v2_behavior()

In [14]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
if not tf.executing_eagerly():
    tf.compat.v1.enable_eager_execution() # Enable eager execution if it's not already enabled
print(tf.executing_eagerly())  # Confirm that eager execution is now enabled

True


In [15]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import RMSprop, Adam
from tensorflow.keras.losses import MeanSquaredError

TEXT = "base_case"
SEED_NUM = 123

In [16]:
u_opt = np.array([
    0.0241,
0.0225,
0.021,
0.0195,
0.0181,
0.0168,
0.0157,
0.0147,
0.0137,
0.0129,
0.0121,
0.0114,
0.0107,
0.0101,
0.0095,
0.009,
0.0085,
0.0081,
0.0077,
0.0074,
0.007,
0.0067,
0.0064,
0.0061,
0.0059,
0.0057,
0.0055,
0.0053,
0.0052,
0.0051,
0.0049,
0.0048,
0.0048,
0.0047,
0.0046,
0.0046,
0.0045,
0.0045,
0.0045,
0.0044,
0.0044,
0.0044,
0.0044,
0.0044,
0.0044,
0.0044,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043,
0.0043])

In [17]:
birth = 0.000053 
death =  0.000033 

alpha = 0.00135
sigma = 0.06352
beta1 = 0.2812 
beta2 = 0.15838 
beta3 = 0.0388 
delta1 = 0.28505
delta2 = 0.28269
delta3 = 0.14206
gamma = 0.30954
p1 = 0.01845
p2 = 0.1431
mu = 0.0042
sigma1 = 0.09275 
sigma2 = 0.03887 
sigma3 = 0.07517 
sigma4 = 0.06302 
sigma5 = 0.07878 
sigma6 = 0.06123 
sigma7 = 0.0611 
sigma8 = 0.06199 

N = 6629870
S0 = 3674149/N 
V0 =2845438/N
E0 = 67789/N
I10 = 12113/N
I20 = 498/N
I30 = 96/N
R0 = 28911/N
D0 = 876/N


In [18]:
c1 = 100 *100
c2 = 20
c3 = 50 
c4 = 200 
c5 = 1000 
c6 = 1/2*300 

In [19]:
k0 =  0.006
k1 = -0.1341

def p1fn(alpha):
    return k0 + k1 * alpha

In [20]:
t_max = 60
points = 60

X = np.random.randint(0, N, size = (points-1, 9)) 
Y = np.zeros((points-1, 1))

dt = t_max/(points)
sqrt_dt = np.sqrt(dt)

### Actual Vacc

In [21]:
act_u = np.array([
  0.0126,0.0132,0.0141,0.0151,0.0158,0.0166,0.0174,0.0179,0.0187,0.0195,0.0201,0.0206,0.0217,0.0227,0.0236,
  0.0248,0.0259,0.0266,0.0268,0.0273,0.0271,0.0268,0.0264,0.0258,0.0252,0.0244,0.024,0.0211,0.0211,0.0207,
  0.0203,0.0195,0.019,0.0179,0.0194,0.0182,0.0173,0.0164,0.0158,0.0154,0.0148,0.0141,0.0135,0.0127,0.0116,
  0.0111,0.0106,0.0108,0.0101,0.0097,0.0093,0.0091,0.0086,0.0087,0.0074,0.007,0.0064,0.0057,0.005,0.0048]) 


np.random.seed(SEED_NUM) #act vacc

repeat_times = points/len(act_u)
repeated_act_u = np.repeat(act_u, repeat_times , axis=0)

S_act = np.zeros(int(t_max/dt) + 1)
V_act = np.zeros(int(t_max/dt) + 1)
E_act = np.zeros(int(t_max/dt) + 1)
I1_act = np.zeros(int(t_max/dt) + 1)
I2_act = np.zeros(int(t_max/dt) + 1)
I3_act = np.zeros(int(t_max/dt) + 1)
R_act = np.zeros(int(t_max/dt) + 1)
D_act = np.zeros(int(t_max/dt) + 1)

S_act[0] = S0 #* N_pob
V_act[0] = V0 #* N_pob
E_act[0] = E0 #* N_pob
I1_act[0] = I10 #* N_pob
I2_act[0] = I20 #* N_pob
I3_act[0] = I30 #* N_pob
R_act[0] = R0 #* N_pob
D_act[0] = D0 #* N_pob

for i in range(1, len(S_act)):
 
    dW1 = np.random.randn()
    dW2 = np.random.randn()
    dW3 = np.random.randn()
    dW4 = np.random.randn()
    dW5 = np.random.randn()
    dW6 = np.random.randn()
    dW7 = np.random.randn()
    dW8 = np.random.randn()
 
    dS = (birth - ( beta1 * I1_act[i-1] + beta2 * I2_act[i-1] + beta3 * I3_act[i-1]) * S_act[i-1] - repeated_act_u[i-1] * S_act[i-1] - death* S_act[i-1]) 
    dV = (repeated_act_u[i-1] * S_act[i-1] - sigma * (beta1 * I1_act[i-1] + beta2 * I2_act[i-1] + beta3 * I3_act[i-1]) * V_act[i-1] - death* V_act[i-1])  
    dE = ((beta1 * I1_act[i-1] + beta2 * I2_act[i-1] + beta3 * I3_act[i-1]) * S_act[i-1]  \
          + sigma * (beta1 * I1_act[i-1] + beta2 * I2_act[i-1] + beta3 * I3_act[i-1]) * V_act[i-1] - gamma * E_act[i-1]- death* E_act[i-1]) 
    dI1 = (gamma * E_act[i-1] - (delta1 + p1fn(repeated_act_u[i-1])) * I1_act[i-1]- death* I1_act[i-1]) 
    dI2 = (p1fn(repeated_act_u[i-1]) * I1_act[i-1] - (delta2 + p2) * I2_act[i-1]- death* I2_act[i-1]) 
    dI3 = (p2 * I2_act[i-1] - (delta3 + mu) * I3_act[i-1]- death* I3_act[i-1]) 
    dR = (delta1 * I1_act[i-1]+ delta2 * I2_act[i-1] + delta3 * I3_act[i-1]- death* R_act[i-1]) 
    dD = (mu * I3_act[i-1])

    S_act[i] = S_act[i-1] + dS * dt + sigma1 * S_act[i-1] * sqrt_dt * dW1 
    V_act[i] = V_act[i-1] + dV * dt + sigma2 * V_act[i-1] * sqrt_dt * dW2
    E_act[i] = E_act[i-1] + dE * dt + sigma3 * E_act[i-1] * sqrt_dt * dW3
    I1_act[i] = I1_act[i-1] + dI1 * dt + sigma4 * I1_act[i-1] * sqrt_dt * dW4
    I2_act[i] = I2_act[i-1] + dI2 * dt + sigma5 * I2_act[i-1] * sqrt_dt * dW5
    I3_act[i] = I3_act[i-1] + dI3 * dt + sigma6 * I3_act[i-1] * sqrt_dt * dW6
    R_act[i] = R_act[i-1] + dR * dt + sigma7 * R_act[i-1] * sqrt_dt * dW7
    D_act[i] = D_act[i-1] + dD * dt + sigma8 * D_act[i-1] * sqrt_dt * dW8


N_path_act = np.repeat(1, len(S_act))
hosp_act_cost = c3 * sum(I1_act*N_path_act) + c4 * sum(I2_act*N_path_act) + c5 * sum(I3_act*N_path_act)
vacc_act_cost = c1 * sum(repeated_act_u**2)
qua_act_cost = c2 * sum(E_act*N_path_act)
econ_act_cost = c6 * sum(E_act*N_path_act+I1_act*N_path_act+I2_act*N_path_act+I3_act*N_path_act+D_act*N_path_act)
total_act_cost = hosp_act_cost + vacc_act_cost + qua_act_cost+econ_act_cost


### No vacc

In [22]:
np.random.seed(SEED_NUM) #no vacc

alpha_ori = 0

S_ori = np.zeros(int(t_max/dt) + 1)
V_ori = np.zeros(int(t_max/dt) + 1)
E_ori = np.zeros(int(t_max/dt) + 1)
I1_ori = np.zeros(int(t_max/dt) + 1)
I2_ori = np.zeros(int(t_max/dt) + 1)
I3_ori = np.zeros(int(t_max/dt) + 1)
R_ori = np.zeros(int(t_max/dt) + 1)
D_ori = np.zeros(int(t_max/dt) + 1)

S_ori[0] = S0 #* N
V_ori[0] = V0 #* N
E_ori[0] = E0 #* N
I1_ori[0] = I10 #* N
I2_ori[0] = I20 #* N
I3_ori[0] = I30 #* N
R_ori[0] = R0 #* N
D_ori[0] = D0 #* N

for i in range(1, len(S_ori)):

    dW1 = np.random.randn()
    dW2 = np.random.randn()
    dW3 = np.random.randn()
    dW4 = np.random.randn()
    dW5 = np.random.randn()
    dW6 = np.random.randn()
    dW7 = np.random.randn()
    dW8 = np.random.randn()

    dS_ori = (birth - ( beta1 * I1_ori[i-1] + beta2 * I2_ori[i-1] + beta3 * I3_ori[i-1]) * S_ori[i-1]- alpha_ori * S_ori[i-1]-death * S_ori[i-1]) 
    dV_ori = (alpha_ori * S_ori[i-1] - sigma * (beta1 * I1_ori[i-1] + beta2 * I2_ori[i-1] + beta3 * I3_ori[i-1]) * V_ori[i-1] - death*V_ori[i-1])  
    dE_ori = ((beta1 * I1_ori[i-1] + beta2 * I2_ori[i-1] + beta3 * I3_ori[i-1]) * S_ori[i-1]  \
          + sigma * (beta1 * I1_ori[i-1] + beta2 * I2_ori[i-1] + beta3 * I3_ori[i-1]) * V_ori[i-1]  - gamma * E_ori[i-1]-death*E_ori[i-1]) 
    dI1_ori = (gamma * E_ori[i-1] - (delta1 + p1) * I1_ori[i-1]-death*I1_ori[i-1]) 
    dI2_ori = (p1 * I1_ori[i-1] - (delta2 + p2) * I2_ori[i-1]-death*I2_ori[i-1]) 
    dI3_ori = (p2 * I2_ori[i-1] - (delta3 + mu) * I3_ori[i-1]-death*I3_ori[i-1]) 
    dR_ori = (delta1 * I1_ori[i-1]+ delta2 * I2_ori[i-1] + delta3 * I3_ori[i-1]-death*R_ori[i-1]) 
    dD_ori = (mu * I3_ori[i-1])

    S_ori[i] = S_ori[i-1] + dS_ori * dt + sigma1 * S_ori[i-1] * sqrt_dt * dW1
    V_ori[i] = V_ori[i-1] + dV_ori * dt + sigma2 * V_ori[i-1] * sqrt_dt * dW2
    E_ori[i] = E_ori[i-1] + dE_ori * dt + sigma3 * E_ori[i-1] * sqrt_dt * dW3
    I1_ori[i] = I1_ori[i-1] + dI1_ori * dt + sigma4 * I1_ori[i-1] * sqrt_dt * dW4
    I2_ori[i] = I2_ori[i-1] + dI2_ori * dt + sigma5 * I2_ori[i-1] * sqrt_dt * dW5
    I3_ori[i] = I3_ori[i-1] + dI3_ori * dt + sigma6 * I3_ori[i-1] * sqrt_dt * dW6
    R_ori[i] = R_ori[i-1] + dR_ori * dt + sigma7 * R_ori[i-1] * sqrt_dt * dW7
    D_ori[i] = D_ori[i-1] + dD_ori * dt + sigma8 * D_ori[i-1] * sqrt_dt * dW8

N_path_ori = np.repeat(1, len(S_act))

hosp_ori_cost = c3 * sum(I1_ori*N_path_ori) + c4 * sum(I2_ori*N_path_ori) + c5 * sum(I3_ori*N_path_ori)
vacc_ori_cost = c1 * sum(np.repeat(alpha_ori**2, points))
qua_ori_cost = c2 * sum(E_ori*N_path_ori)
econ_ori_cost = c6 * sum(E_ori*N_path_ori + I1_ori*N_path_ori+I2_ori*N_path_ori+I3_ori*N_path_ori + D_ori*N_path_ori)

total_ori_cost = hosp_ori_cost+vacc_ori_cost+qua_ori_cost+econ_ori_cost


### Constant vacc

In [23]:
np.random.seed(SEED_NUM) #constant vacc
alpha_constant = np.average(act_u)
S_con = np.zeros(int(t_max/dt) + 1)
V_con = np.zeros(int(t_max/dt) + 1)
E_con = np.zeros(int(t_max/dt) + 1)
I1_con = np.zeros(int(t_max/dt) + 1)
I2_con = np.zeros(int(t_max/dt) + 1)
I3_con = np.zeros(int(t_max/dt) + 1)
R_con = np.zeros(int(t_max/dt) + 1)
D_con = np.zeros(int(t_max/dt) + 1)

S_con[0] = S0 #* N
V_con[0] = V0 #* N
E_con[0] = E0 #* N
I1_con[0] = I10 #* N
I2_con[0] = I20 #* N
I3_con[0] = I30 #* N
R_con[0] = R0 #* N
D_con[0] = D0 #* N


for i in range(1, len(S_con)):

    dW1 = np.random.randn()
    dW2 = np.random.randn()
    dW3 = np.random.randn()
    dW4 = np.random.randn()
    dW5 = np.random.randn()
    dW6 = np.random.randn()
    dW7 = np.random.randn()
    dW8 = np.random.randn()

    dS_con = (birth - ( beta1 * I1_con[i-1] + beta2 * I2_con[i-1] + beta3 * I3_con[i-1]) * S_con[i-1]- alpha_constant * S_con[i-1]-death * S_con[i-1]) 
    dV_con = (alpha_constant * S_con[i-1] - sigma * (beta1 * I1_con[i-1] + beta2 * I2_con[i-1] + beta3 * I3_con[i-1]) * V_con[i-1] - death*V_con[i-1])  
    dE_con = ((beta1 * I1_con[i-1] + beta2 * I2_con[i-1] + beta3 * I3_con[i-1]) * S_con[i-1]  \
          + sigma * (beta1 * I1_con[i-1] + beta2 * I2_con[i-1] + beta3 * I3_con[i-1]) * V_con[i-1]  - gamma * E_con[i-1]-death*E_con[i-1]) 
    dI1_con = (gamma * E_con[i-1] - (delta1 + p1) * I1_con[i-1]-death*I1_con[i-1]) 
    dI2_con = (p1 * I1_con[i-1] - (delta2 + p2) * I2_con[i-1]-death*I2_con[i-1]) 
    dI3_con = (p2 * I2_con[i-1] - (delta3 + mu) * I3_con[i-1]-death*I3_con[i-1]) 
    dR_con = (delta1 * I1_con[i-1]+ delta2 * I2_con[i-1] + delta3 * I3_con[i-1]-death*R_con[i-1]) 
    dD_con = (mu * I3_con[i-1])

    S_con[i] = S_con[i-1] + dS_con * dt + sigma1 * S_con[i-1] * sqrt_dt * dW1
    V_con[i] = V_con[i-1] + dV_con * dt + sigma2 * V_con[i-1] * sqrt_dt * dW2
    E_con[i] = E_con[i-1] + dE_con * dt + sigma3 * E_con[i-1] * sqrt_dt * dW3
    I1_con[i] = I1_con[i-1] + dI1_con * dt + sigma4 * I1_con[i-1] * sqrt_dt * dW4
    I2_con[i] = I2_con[i-1] + dI2_con * dt + sigma5 * I2_con[i-1] * sqrt_dt * dW5
    I3_con[i] = I3_con[i-1] + dI3_con * dt + sigma6 * I3_con[i-1] * sqrt_dt * dW6
    R_con[i] = R_con[i-1] + dR_con * dt + sigma7 * R_con[i-1] * sqrt_dt * dW7
    D_con[i] = D_con[i-1] + dD_con * dt + sigma8 * D_con[i-1] * sqrt_dt * dW8

N_path_con = np.repeat(1, len(S_act))
hosp_constant_cost = c3 * sum(I1_con*N_path_con) + c4 * sum(I2_con*N_path_con) + c5 * sum(I3_con*N_path_con)
vacc_constant_cost = c1 * sum(np.repeat(alpha_constant**2, points))
qua_constant_cost = c2 * sum(E_con*N_path_con)
econ_constant_cost = c6 * sum(E_con*N_path_con+I1_con*N_path_con+I2_con*N_path_con+I3_con*N_path_con+D_con*N_path_con)

total_constant_cost = hosp_constant_cost+vacc_constant_cost+qua_constant_cost+econ_constant_cost

### Current Optimal Vaccination

In [24]:
# optimal vaccination rate use past paper's result
repeat_times = points/len(u_opt)
repeated_u = np.repeat(u_opt, repeat_times , axis=0)


In [25]:
np.random.seed(123) #optimal vacc seed

S = np.zeros(int(t_max/dt) + 1)
V = np.zeros(int(t_max/dt) + 1)
E = np.zeros(int(t_max/dt) + 1)
I1 = np.zeros(int(t_max/dt) + 1)
I2 = np.zeros(int(t_max/dt) + 1)
I3 = np.zeros(int(t_max/dt) + 1)
R = np.zeros(int(t_max/dt) + 1)
D = np.zeros(int(t_max/dt) + 1)

S[0] = S0 #* N_pob
V[0] = V0 #* N_pob
E[0] = E0 #* N_pob
I1[0] = I10 #* N_pob
I2[0] = I20 #* N_pob
I3[0] = I30 #* N_pob
R[0] = R0 #* N_pob
D[0] = D0 #* N_pob

for i in range(1, len(S)):
 
    dW1 = np.random.randn()
    dW2 = np.random.randn()
    dW3 = np.random.randn()
    dW4 = np.random.randn()
    dW5 = np.random.randn()
    dW6 = np.random.randn()
    dW7 = np.random.randn()
    dW8 = np.random.randn()
 
    dS = (birth - ( beta1 * I1[i-1] + beta2 * I2[i-1] + beta3 * I3[i-1]) * S[i-1] - repeated_u[i-1] * S[i-1] - death* S[i-1]) 
    dV = (repeated_u[i-1] * S[i-1] - sigma * (beta1 * I1[i-1] + beta2 * I2[i-1] + beta3 * I3[i-1]) * V[i-1] - death* V[i-1])  
    dE = ((beta1 * I1[i-1] + beta2 * I2[i-1] + beta3 * I3[i-1]) * S[i-1]  \
          + sigma * (beta1 * I1[i-1] + beta2 * I2[i-1] + beta3 * I3[i-1]) * V[i-1] - gamma * E[i-1]- death* E[i-1]) 
    dI1 = (gamma * E[i-1] - (delta1 + p1fn(repeated_u[i-1])) * I1[i-1]- death* I1[i-1]) 
    dI2 = (p1fn(repeated_u[i-1]) * I1[i-1] - (delta2 + p2) * I2[i-1]- death* I2[i-1]) 
    dI3 = (p2 * I2[i-1] - (delta3 + mu) * I3[i-1]- death* I3[i-1]) 
    dR = (delta1 * I1[i-1]+ delta2 * I2[i-1] + delta3 * I3[i-1]- death* R[i-1]) 
    dD = (mu * I3[i-1])

    S[i] = S[i-1] + dS * dt + sigma1 * S[i-1] * sqrt_dt * dW1 
    V[i] = V[i-1] + dV * dt + sigma2 * V[i-1] * sqrt_dt * dW2
    E[i] = E[i-1] + dE * dt + sigma3 * E[i-1] * sqrt_dt * dW3
    I1[i] = I1[i-1] + dI1 * dt + sigma4 * I1[i-1] * sqrt_dt * dW4
    I2[i] = I2[i-1] + dI2 * dt + sigma5 * I2[i-1] * sqrt_dt * dW5
    I3[i] = I3[i-1] + dI3 * dt + sigma6 * I3[i-1] * sqrt_dt * dW6
    R[i] = R[i-1] + dR * dt + sigma7 * R[i-1] * sqrt_dt * dW7
    D[i] = D[i-1] + dD * dt + sigma8 * D[i-1] * sqrt_dt * dW8
    

In [26]:
# cost table
S_opt = S
V_opt = V
E_opt = E
I1_opt = I1
I2_opt = I2
I3_opt = I3
R_opt = R
D_opt = D
N_path_opt = np.repeat(1, len(S_opt))
hosp_opt_cost = c3 * sum(I1*N_path_opt) + c4 * sum(I2*N_path_opt) + c5 * sum(I3*N_path_opt)
vacc_opt_cost = c1 * sum(repeated_u**2)
qua_opt_cost = c2 * sum(E*N_path_opt)
econ_opt_cost = c6 * sum(E*N_path_opt+I1*N_path_opt+I2*N_path_opt+I3*N_path_opt+D*N_path_opt)
total_opt_cost = hosp_opt_cost+vacc_opt_cost+qua_opt_cost+econ_opt_cost


### Summary Table - Cost Analysis

In [27]:
summary_table= {'Type': ['Policy', 'Hosp', 'Econ'],
                'ori': [vacc_ori_cost+qua_ori_cost, hosp_ori_cost, econ_ori_cost],
                'constant': [vacc_constant_cost+qua_constant_cost, hosp_constant_cost, econ_constant_cost],
                'actual': [vacc_act_cost+qua_act_cost,hosp_act_cost,econ_act_cost],
                'opt': [vacc_opt_cost+qua_opt_cost,hosp_opt_cost,econ_opt_cost]
               }
df_summary = pd.DataFrame(summary_table)
print(df_summary.to_string(index=False))

  Type       ori   constant     actual       opt
Policy  2.113271 172.912451 197.960731 53.604848
  Hosp 10.949096   8.607654   5.671156  6.028926
  Econ 34.829766  27.094706  27.257578 28.067588


#### NO vaccination

In [28]:
print(vacc_ori_cost+qua_ori_cost)
print(hosp_ori_cost)
print(econ_ori_cost)

2.1132709076920526
10.9490956837207
34.82976625418565


#### Constant vaccination

In [29]:
print(vacc_constant_cost+qua_constant_cost)
print(hosp_constant_cost)
print(econ_constant_cost)

172.9124513237049
8.60765428683954
27.09470623719303


#### Actual vaccination

In [30]:
print(vacc_act_cost+qua_act_cost)
print(hosp_act_cost)
print(econ_act_cost)

197.96073142083594
5.6711556077056695
27.25757775015591


#### opt vaccination

In [31]:
print(vacc_opt_cost+qua_opt_cost)
print(hosp_opt_cost)
print(econ_opt_cost)

53.604847701065694
6.0289259000311475
28.067588058870832
